In [1]:
# %pip install transformers datasets torch scikit-learn evaluate

In [ ]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from model import *
from transformers import TrainingArguments
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding

In [9]:
sys.path.append('.')

In [11]:

def main_function(mode="multiclass"):
    if mode == "binary":
        num_labels = 2
        model_class = BertweetModelBinary
        train_path = "\data\processed_data\test_binary_preprocessed.csv"
        val_path = "\data\processed_data\validation_binary_preprocessed.csv"
        test_path = "\data\processed_data\test_binary_preprocessed.csv"
        output_dir = "./results_binary"
        logging_dir = "./logs_binary"
    else:
        num_labels = 5
        model_class = BertweetModelMulticlass
        train_path = "\data\processed_data\train_multiclass_balanced.csv"
        val_path = "\data\processed_data\validation_multiclass_preprocessed.csv"
        test_path = "\data\processed_data\test_multiclass_preprocessed.csv"
        output_dir = "./results_multiclass"
        logging_dir = "./logs_multiclass"

    
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)


    train_df = train_df.dropna(subset=["tweet_soft"])
    val_df = val_df.dropna(subset=["tweet_soft"])
    test_df = test_df.dropna(subset=["tweet_soft"])

    ## Convert to huggingface dataset
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)
    test_dataset = Dataset.from_pandas(test_df)


    ## Model initialization
    classifier = model_class(num_labels=num_labels, model_name="vinai/bertweet-base")

    print("Device being used:", classifier.device)
    print(f"model {classifier}")


    # Preprocess data
    tokenized_train_dataset = classifier.preprocess_data(train_dataset)
    tokenized_val_dataset = classifier.preprocess_data(val_dataset)
    tokenized_test_dataset = classifier.preprocess_data(test_dataset)

    # print("Tokenizers: ",tokenized_train_dataset.unique("label"))

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,  
        per_device_train_batch_size=16, 
        per_device_eval_batch_size=16,
        num_train_epochs=5,  
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_dir=logging_dir,
        logging_strategy="epoch",   
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_weighted",
        fp16=True,
        warmup_ratio=0.1,
        logging_first_step=True,
        greater_is_better=True,
        report_to="none",
        gradient_accumulation_steps=2,  
        max_grad_norm=1.0,  
        dataloader_drop_last=False,
        lr_scheduler_type="cosine_with_restarts",
    )

     # early stopping callback
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.001
    )

    data_collator = DataCollatorWithPadding(tokenizer=classifier.tokenizer)

    # Train with callback
    classifier.train(
        tokenized_train_dataset,
        tokenized_val_dataset,
        training_args,
        callbacks=[early_stopping],
        data_collator=data_collator
    )

    # Evaluation
    classifier.plot_confusion_matrix(tokenized_val_dataset)
    eval_results = classifier.evaluate(tokenized_val_dataset)
    classifier.print_metrics_summary(eval_results)

    classifier.save_model(output_dir)

    return classifier, eval_results

In [ ]:
main_function("binary")